# Fine-tune a pretrained language model with TrainLM

Use this when you already have pretrained weights. TrainLM loads the model and tokenizer, prepares a plain-text dataset, fine-tunes, evaluates, and saves the model locally. This is different from pretraining from scratch, which starts with random weights.

Run the cells from top to bottom. The defaults use a tiny GPT-2 model and three steps as a workflow smoke test; they do not measure model quality. Change the model and training budget in the settings cell for a real run. Nothing is uploaded to the Hub.

Native TorchTPU is experimental. Select it explicitly in the settings cell; this notebook does not claim TPU training has been certified.

In [ ]:
#@title 1. Install TrainLM without replacing Torch
from importlib import metadata, util
import subprocess
import sys

# This notebook is on the same feature branch as the API. After it merges,
# change this to "main".
TRAINLM_REF = "feat/01-pretrained-finetuning-api"
trainlm_source = (
    "git+https://github.com/Dhiraj309/TrainLM.git@" + TRAINLM_REF
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "--no-deps", trainlm_source,
    ],
    check=True,
)

# Install only the optional dataset package if the runtime does not have it.
if util.find_spec("datasets") is None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--quiet",
            "--disable-pip-version-check", "datasets>=3,<5",
        ],
        check=True,
    )

import torch
import transformers
import datasets
print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"datasets={metadata.version('datasets')}")

## 2. Choose the model, dataset, and run size

The dataset revision must be an immutable 40-character commit SHA, and the dataset must have separate training and validation splits. The example is intentionally small. For useful training, choose a model that fits your runtime and increase `MAX_STEPS` after the smoke run succeeds.

To resume, set `RESUME_FROM_CHECKPOINT = "latest"` and raise `MAX_STEPS` above the saved step. TrainLM restores the model, dataset, and training settings from `OUTPUT_DIR/finetuning-summary.json`, so you do not need to enter them again. Keep `OUTPUT_DIR` on persistent storage to survive a runtime reset. If the summary is missing or incompatible, the notebook explains how to recover and skips training and download cells.

For native TorchTPU, select `native_tpu` and use a TPU runtime. This path is experimental; `auto` does not select TPU yet.

In [ ]:
#@title 3. Edit these run settings
from pathlib import Path

from trainlm import (
    TrainLMFineTuningConfig,
    TrainLMTrainingArguments,
    restore_finetuning_config,
    run_finetuning,
)

MODEL_NAME_OR_PATH = "sshleifer/tiny-gpt2" #@param {type:"string"}
MODEL_REVISION = "" #@param {type:"string"}
DATASET_ID = "roneneldan/TinyStories" #@param {type:"string"}
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64" #@param {type:"string"}
TEXT_FIELD = "text" #@param {type:"string"}
TRAIN_SPLIT = "train" #@param {type:"string"}
VALIDATION_SPLIT = "validation" #@param {type:"string"}
MAX_STEPS = 3 #@param {type:"integer"}
SAVE_STEPS = 100 #@param {type:"integer"}
RESUME_FROM_CHECKPOINT = "" #@param ["", "latest"]
SEQUENCE_LENGTH = 128 #@param {type:"integer"}
BATCH_SIZE = 2 #@param {type:"integer"}
LEARNING_RATE = 2e-5 #@param {type:"number"}
ACCELERATOR = "cpu" #@param ["cpu", "cuda", "native_tpu"]
OUTPUT_DIR = Path("runs/trainlm-finetuning")

if RESUME_FROM_CHECKPOINT == "latest":
    try:
        config = restore_finetuning_config(
            OUTPUT_DIR / "finetuning-summary.json",
            max_steps=MAX_STEPS,
        )
    except ValueError as error:
        config = None
        print(f"resume_status=blocked error={type(error).__name__}")
        print(f"resume_reason={error}")
        print(
            "resume_recovery=point OUTPUT_DIR to the previous run folder with its "
            "summary and checkpoints; use the matching TrainLM version; increase "
            "MAX_STEPS; or clear RESUME_FROM_CHECKPOINT to start a new run."
        )
else:
    config = TrainLMFineTuningConfig(
        model_name_or_path=MODEL_NAME_OR_PATH,
        model_revision=MODEL_REVISION or None,
        dataset_id=DATASET_ID,
        dataset_revision=DATASET_REVISION,
        text_field=TEXT_FIELD,
        train_split=TRAIN_SPLIT,
        validation_split=VALIDATION_SPLIT,
        args=TrainLMTrainingArguments(
            output_dir=OUTPUT_DIR,
            max_steps=MAX_STEPS,
            sequence_length=SEQUENCE_LENGTH,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            logging_steps=1,
            save_steps=SAVE_STEPS,
            accelerator=ACCELERATOR,
            bf16=ACCELERATOR == "native_tpu",
        ),
    )

if config is not None:
    print(f"model={config.model_name_or_path}")
    print(f"dataset={config.dataset_id}@{config.dataset_revision}")
    print(
        f"steps={config.args.max_steps}, save_steps={config.args.save_steps}, "
        f"resume={RESUME_FROM_CHECKPOINT or 'no'}, "
        f"sequence_length={config.args.sequence_length}, "
        f"accelerator={config.args.accelerator}"
    )


## 3. Fine-tune and evaluate

TrainLM keeps the training loop and padding-label handling inside the API. This call evaluates on the configured validation split and writes the model and tokenizer to `OUTPUT_DIR / "model"`.

In [ ]:
if config is None:
    result = None
    print("run_status=skipped; resolve resume setup before rerunning")
else:
    try:
        result = run_finetuning(
            config,
            resume_from_checkpoint=RESUME_FROM_CHECKPOINT or None,
        )
    except Exception as error:
        result = None
        print(f"run_status=failed error_type={type(error).__name__}")
        print(f"run_summary={OUTPUT_DIR / 'finetuning-summary.json'}")
    else:
        print(result.summary())


## 4. Download the model

TrainLM writes a JSON run summary beside the model export with the settings, resolved revisions, and metrics. The final cell bundles the model only after successful training; on failure it downloads the summary without including a stale model. Colab starts a download, and Kaggle shows the output path.

In [ ]:
if config is None:
    print("bundle_status=skipped; resolve resume setup before downloading")
else:
    import json
    from pathlib import Path
    from zipfile import ZIP_DEFLATED, ZipFile

    summary_path = OUTPUT_DIR / "finetuning-summary.json"
    if not summary_path.is_file():
        raise RuntimeError("TrainLM did not write the fine-tuning summary.")
    report = json.loads(summary_path.read_text(encoding="utf-8"))
    bundle_path = OUTPUT_DIR / "finetuned-model.zip"
    with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as bundle:
        model_path = Path(report["model_path"])
        if report["status"] == "completed" and model_path.is_dir():
            for path in model_path.rglob("*"):
                if path.is_file():
                    bundle.write(path, path.relative_to(model_path.parent))
        bundle.write(summary_path, summary_path.name)
    print(f"run_status={report['status']}")

    try:
        from google.colab import files
    except ImportError:
        print(f"Kaggle or local runtime: download {bundle_path}")
    else:
        files.download(str(bundle_path))
